# EN3150 Assignment 03 — Part 5: Fine-tuning ShuffleNetV2

## 1. Setup
ShuffleNetV2 uses ImageNet normalization for pretrained weights. The same images and train/validation/test split are used as Models A and B. Torchvision automatically downloads the pretrained weights on first use (internet required).

In [1]:
from a03_common import *
from torchvision import models

FREEZE_EPOCHS = 3        # phase 1: train the new head only
FINETUNE_EPOCHS = 12     # phase 2: fine-tune the whole network
HEAD_LR, BACKBONE_LR = 1e-3, 2e-4

set_seed(SEED)
print("Device:", DEVICE)
data = Data(norm="imagenet", batch_size=64, augment=True)
print("Train/Val/Test:", len(data.idx["train"]), len(data.idx["val"]), len(data.idx["test"]))

Device: cpu
Train/Val/Test: 18900 4050 4050


## 2. Model: ShuffleNetV2 (x1.0)
Uses channel splitting, channel shuffling, and depthwise convolutions for efficient inference.
Focuses on real-device speed, considering memory access and hardware limitations alongside MACs.
Replaces the original ImageNet classifier with a 10-class output layer for EuroSAT.

In [2]:
weights = models.ShuffleNet_V2_X1_0_Weights.IMAGENET1K_V1
model = models.shufflenet_v2_x1_0(weights=weights)
model.fc = nn.Linear(model.fc.in_features, data.num_classes)                         # new 10-class head
model = model.to(DEVICE)

total, trainable = count_params(model)
print(f"ShuffleNetV2 x1.0 | total params: {total:,} | trainable: {trainable:,}")
print(model.fc)

Downloading: "https://download.pytorch.org/models/shufflenetv2_x1-5666bf0f80.pth" to C:\Users\Asus/.cache\torch\hub\checkpoints\shufflenetv2_x1-5666bf0f80.pth


100.0%


ShuffleNetV2 x1.0 | total params: 1,263,854 | trainable: 1,263,854
Linear(in_features=1024, out_features=10, bias=True)


## 3. Fine-tuning strategy (two phases)
1. **Phase 1 - head only (3 epochs, Adam lr = 1e-3):** the new classifier layer starts random. Training it alone first stops large random gradients from damaging the good pretrained features.
2. **Phase 2 - whole network (12 epochs, Adam):** unfreeze everything with a **small learning rate for the pretrained backbone (2e-4)** and a larger one for the head (1e-3), so the features adapt gently to satellite images.

In [ ]:
HEAD = "fc"

def set_trainable(model, backbone):
    for n, p in model.named_parameters():
        p.requires_grad = n.startswith(HEAD) or backbone

total, _ = count_params(model)

# ---- phase 1: head only ----
set_trainable(model, backbone=False)
_, trainable = count_params(model)
print(f"Phase 1: total params {total:,} | trainable {trainable:,}")
opt1 = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=HEAD_LR)
hist = train_model(model, opt1, data, FREEZE_EPOCHS, name="ShuffleNetV2 phase 1")

# ---- phase 2: whole network ----
set_trainable(model, backbone=True)
_, trainable = count_params(model)
print(f"\nPhase 2: total params {total:,} | trainable {trainable:,}")
backbone = [p for n, p in model.named_parameters() if not n.startswith(HEAD)]
head = [p for n, p in model.named_parameters() if n.startswith(HEAD)]
opt2 = torch.optim.Adam([{"params": backbone, "lr": BACKBONE_LR}, {"params": head, "lr": HEAD_LR}])
hist = train_model(model, opt2, data, FINETUNE_EPOCHS, name="ShuffleNetV2 phase 2", hist=hist)

## 4. Learning curves
The dashed line marks the end of phase 1. A visible drop in validation loss right after it is the benefit of unfreezing the backbone.

In [ ]:
plot_history(hist, "ShuffleNetV2", "shufflenetv2_curves.png", marks=[FREEZE_EPOCHS])

## 5. Test-set evaluation

In [ ]:
ev = evaluate(model, data)
cm = plot_confusion(ev["y_true"], ev["y_pred"], data.classes,
                    f"ShuffleNetV2 - confusion matrix (test acc {ev['acc']:.2%})", "shufflenetv2_confusion.png")

## 6. Parameter count, model size (MB) and cost metrics
Model size is the float32 `state_dict` on disk. **Time per epoch** is reported for phase 2 (full backpropagation through the whole network), the figure comparable with the custom models. Everything goes to the results JSON used by the final comparison notebook.

In [ ]:
total, trainable = count_params(model)
ft_time = float(np.mean(hist["epoch_time"][FREEZE_EPOCHS:]))
res = save_results("shufflenetv2", "ShuffleNetV2 (fine-tuned)", model, hist, ev,
                   "Adam, 2-phase fine-tuning", data, time_per_epoch_s=ft_time,
                   extra={"time_per_epoch_phase1_s": float(np.mean(hist["epoch_time"][:FREEZE_EPOCHS]))})
print(f"\nTotal parameters: {total:,} | Model size: {res['size_kb']/1024:.2f} MB")

## 7. What to write in the report
- **Part 5 table row:** total parameters, model size (MB), test accuracy, macro precision and recall, time per epoch (phase 2), MACs, CPU latency (all printed above).
- **Interpretation:** how much accuracy the pretrained model gains over your from-scratch models and where it still confuses classes (confusion-matrix pairs).
- **Cost side:** compare parameters, MB and MACs against Model B in the final comparison notebook; the pretrained network is many times larger and costlier, so the question is whether the accuracy gain justifies it.